# Day 076 — Exercise 5: ScreenAgent

**What you'll build:** `ScreenAgent` — stateful screen-understanding assistant that wraps all vision tools with history tracking.

**Why it matters:** The class binds all three injections at construction so callers use `agent.describe()` rather than passing mock functions on every call — the same pattern as ImageProcessor, AudioTranscriber, and TalkingHeadPipeline.

In [ ]:
from PIL import Image as _PILImage

def _make_mock_image(width=100, height=100, color=(100, 100, 100)):
    return _PILImage.new('RGB', (width, height), color=color)
_mock_screenshot_fn = lambda region=None: _make_mock_image()
_mock_analyze_fn    = lambda img, q: 'MOCK:' + q[:16]
_mock_llm_fn        = lambda prompt: 'TASK:' + prompt[:12]
def capture_screenshot(region=None, screenshot_fn=None):
    if screenshot_fn is not None:
        return screenshot_fn(region)
    from PIL import ImageGrab
    return ImageGrab.grab(bbox=region)
import io, base64

def analyze_screenshot(image, question, analyze_fn=None):
    if analyze_fn is not None:
        return analyze_fn(image, question)
    import ollama
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': question, 'images': [img_b64]}],
    )
    return resp['message']['content']

def describe_screen(image, analyze_fn=None):
    return analyze_screenshot(
        image, 'Describe what you see on this screen in detail.',
        analyze_fn=analyze_fn)

def read_screen_text(image, analyze_fn=None):
    return analyze_screenshot(
        image, 'Extract all visible text from this image exactly as it appears.',
        analyze_fn=analyze_fn)

def find_elements(image, element_type, analyze_fn=None):
    question = (f'List all {element_type} elements visible in this screenshot. '
                'Be specific about their labels, text, or content.')
    return analyze_screenshot(image, question, analyze_fn=analyze_fn)

def answer_about_screen(image, question, analyze_fn=None):
    return analyze_screenshot(image, question, analyze_fn=analyze_fn)
def run_screen_task(image, task, analyze_fn=None, llm_fn=None):
    description = describe_screen(image, analyze_fn=analyze_fn)
    lines = [
        'You are a screen-reading assistant.',
        'Here is what is visible on screen:',
        '',
        description,
        '',
        f'Task: {task}',
        '',
        'Answer based only on what is visible on screen.',
    ]
    context_prompt = '\n'.join(lines)
    if llm_fn is not None:
        answer = llm_fn(context_prompt)
    else:
        import ollama
        resp = ollama.chat(
            model='llama3.2',
            messages=[{'role': 'user', 'content': context_prompt}],
        )
        answer = resp['message']['content']
    return {'description': description, 'answer': answer, 'task': task}


## Task

Implement `ScreenAgent(screenshot_fn=None, analyze_fn=None, llm_fn=None)`:

- `__init__`: store all 3. `_last_image=None`, `_history=[]`
- `capture(region=None)`: `capture_screenshot(region, self._screenshot_fn)` → store `_last_image` → append `{action:'capture', region:region}` → return img
- `describe/read_text/ask/find`: `img = image if image is not None else self._last_image`; raise ValueError if None; call module fn; append history entry; return result
  - `describe`: `{action:'describe', result:result}`
  - `ask`: `{action:'ask', question:question, result:result}`
  - `find`: `{action:'find', element_type:element_type, result:result}`
- `run(task, image=None)`: if no image, call `self.capture()`; `run_screen_task(img, task, analyze_fn=self._analyze_fn, llm_fn=self._llm_fn)`; append `{action:'run', task:task, result:result['answer']}`
- `history()`: `return list(self._history)`
- `clear_history()`: `self._history.clear()`

## Your Implementation

In [ ]:
class ScreenAgent:
    """Stateful screen-understanding assistant with history tracking."""

    def __init__(self, screenshot_fn=None, analyze_fn=None, llm_fn=None):
        raise NotImplementedError

    def capture(self, region=None):
        raise NotImplementedError

    def describe(self, image=None):
        raise NotImplementedError

    def read_text(self, image=None):
        raise NotImplementedError

    def ask(self, question, image=None):
        raise NotImplementedError

    def find(self, element_type, image=None):
        raise NotImplementedError

    def run(self, task, image=None):
        raise NotImplementedError

    def history(self):
        raise NotImplementedError

    def clear_history(self):
        raise NotImplementedError


In [ ]:
class ScreenAgent:
    def __init__(self, screenshot_fn=None, analyze_fn=None, llm_fn=None):
        self._screenshot_fn = screenshot_fn
        self._analyze_fn = analyze_fn
        self._llm_fn = llm_fn
        self._last_image = None
        self._history = []

    def capture(self, region=None):
        img = capture_screenshot(region=region, screenshot_fn=self._screenshot_fn)
        self._last_image = img
        self._history.append({'action': 'capture', 'region': region})
        return img

    def describe(self, image=None):
        img = image if image is not None else self._last_image
        if img is None:
            raise ValueError('No image: call capture() first or pass image.')
        result = describe_screen(img, analyze_fn=self._analyze_fn)
        self._history.append({'action': 'describe', 'result': result})
        return result

    def read_text(self, image=None):
        img = image if image is not None else self._last_image
        if img is None:
            raise ValueError('No image: call capture() first or pass image.')
        result = read_screen_text(img, analyze_fn=self._analyze_fn)
        self._history.append({'action': 'read_text', 'result': result})
        return result

    def ask(self, question, image=None):
        img = image if image is not None else self._last_image
        if img is None:
            raise ValueError('No image: call capture() first or pass image.')
        result = answer_about_screen(img, question, analyze_fn=self._analyze_fn)
        self._history.append({'action': 'ask', 'question': question, 'result': result})
        return result

    def find(self, element_type, image=None):
        img = image if image is not None else self._last_image
        if img is None:
            raise ValueError('No image: call capture() first or pass image.')
        result = find_elements(img, element_type, analyze_fn=self._analyze_fn)
        self._history.append({'action': 'find', 'element_type': element_type, 'result': result})
        return result

    def run(self, task, image=None):
        img = image if image is not None else self._last_image
        if img is None:
            img = self.capture()
        result = run_screen_task(img, task, analyze_fn=self._analyze_fn, llm_fn=self._llm_fn)
        self._history.append({'action': 'run', 'task': task, 'result': result['answer']})
        return result

    def history(self):
        return list(self._history)

    def clear_history(self):
        self._history.clear()


## Automated checks

In [ ]:

score, total = 0, 6
try:
    from PIL import Image as PILImage

    agent = ScreenAgent(
        screenshot_fn=_mock_screenshot_fn,
        analyze_fn=_mock_analyze_fn,
        llm_fn=_mock_llm_fn,
    )

    img = agent.capture()
    assert isinstance(img, PILImage.Image)
    score += 1; print("✅ capture() returns PIL Image")

    d = agent.describe()
    assert isinstance(d, str)
    score += 1; print("✅ describe() returns str using stored image")

    a = agent.ask('What is this?')
    assert isinstance(a, str)
    score += 1; print("✅ ask() returns str")

    f = agent.find('button')
    assert isinstance(f, str)
    score += 1; print("✅ find() returns str")

    r = agent.run('Identify the app')
    assert isinstance(r, dict) and 'description' in r and 'answer' in r
    score += 1; print("✅ run() returns dict with description and answer")

    hist = agent.history()
    assert isinstance(hist, list) and len(hist) >= 4
    agent.clear_history()
    assert agent.history() == []
    score += 1; print("✅ history() and clear_history() work correctly")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
class ScreenAgent:
    def __init__(self, screenshot_fn=None, analyze_fn=None, llm_fn=None):
        self._screenshot_fn = screenshot_fn
        self._analyze_fn = analyze_fn
        self._llm_fn = llm_fn
        self._last_image = None
        self._history = []

    def capture(self, region=None):
        img = capture_screenshot(region=region, screenshot_fn=self._screenshot_fn)
        self._last_image = img
        self._history.append({'action': 'capture', 'region': region})
        return img

    def describe(self, image=None):
        img = image if image is not None else self._last_image
        if img is None:
            raise ValueError('No image: call capture() first or pass image.')
        result = describe_screen(img, analyze_fn=self._analyze_fn)
        self._history.append({'action': 'describe', 'result': result})
        return result

    def read_text(self, image=None):
        img = image if image is not None else self._last_image
        if img is None:
            raise ValueError('No image: call capture() first or pass image.')
        result = read_screen_text(img, analyze_fn=self._analyze_fn)
        self._history.append({'action': 'read_text', 'result': result})
        return result

    def ask(self, question, image=None):
        img = image if image is not None else self._last_image
        if img is None:
            raise ValueError('No image: call capture() first or pass image.')
        result = answer_about_screen(img, question, analyze_fn=self._analyze_fn)
        self._history.append({'action': 'ask', 'question': question, 'result': result})
        return result

    def find(self, element_type, image=None):
        img = image if image is not None else self._last_image
        if img is None:
            raise ValueError('No image: call capture() first or pass image.')
        result = find_elements(img, element_type, analyze_fn=self._analyze_fn)
        self._history.append({'action': 'find', 'element_type': element_type, 'result': result})
        return result

    def run(self, task, image=None):
        img = image if image is not None else self._last_image
        if img is None:
            img = self.capture()
        result = run_screen_task(img, task, analyze_fn=self._analyze_fn, llm_fn=self._llm_fn)
        self._history.append({'action': 'run', 'task': task, 'result': result['answer']})
        return result

    def history(self):
        return list(self._history)

    def clear_history(self):
        self._history.clear()
```

**Why `image if image is not None else self._last_image`** and not `image or self._last_image`? A real PIL Image is truthy, so both work here — but the `is not None` form correctly handles the case where someone passes a 0×0 image (falsy) as an explicit override.

</details>